In [1]:
# pip install mygene

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

In [2]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [3]:
df.head()

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN


In [4]:
df_genes = df["gene_symbol"]

In [5]:
df_genes

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1053     INPP5F
1054       RIT2
1055       GCH1
1056    SIPA1L2
1057    TMPRSS9
Name: gene_symbol, Length: 1058, dtype: object

In [6]:
df_genes_t = df_genes.dropna()

In [7]:
df_genes_t = df_genes_t.reset_index(drop=False)["gene_symbol"]

In [8]:
df_genes_t

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1051     INPP5F
1052       RIT2
1053       GCH1
1054    SIPA1L2
1055    TMPRSS9
Name: gene_symbol, Length: 1056, dtype: object

In [9]:
df_genes_t.head(20)

0              GPR126
1               SYT11
2             SLC2A13
3             SLC2A13
4             SLC2A13
5             SLC2A15
6     LINC02471;LRRK2
7               LRRK2
8               LRRK2
9         GPRIN3;SNCA
10               MAPT
11    LINC02210-CRHR1
12        GPRIN3;SNCA
13    LINC02210-CRHR1
14           SERPINA1
15           SERPINA1
16        S1PR1;OLFM3
17               MAPT
18               MAPT
19             SPPL2C
Name: gene_symbol, dtype: object

In [10]:
def extrae_gene_symbols(dataframe):
    
    df_genes = dataframe["gene_symbol"]
    
    df_genes = df_genes.dropna()
    
    df_genes = df_genes.reset_index(drop=False)["gene_symbol"]
    
    lista_symbols = []
    
    for i, gene in enumerate(df_genes):
        
        gene = gene.replace(",", ";")
    
        if "dist" in gene:
            continue

        elif ";" in gene:

            separacion1 = gene.split(";")

            for gen in separacion1:
                lista_symbols.append(gen)

        else:
            lista_symbols.append(gene)
            
    return lista_symbols

In [11]:
lista_symbols = extrae_gene_symbols(df)

In [12]:
# for gen in lista_symbols:
#     print(gen)

In [13]:
mg = mygene.MyGeneInfo()

In [20]:
gen = ["GPR126"]
resultado_prueba = mg.querymany(gen, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [21]:
print(resultado_prueba)

[{'query': 'GPR126', '_id': '57211', '_score': 18.989347, 'genomic_pos_hg19': {'chr': '6', 'end': 142767403, 'start': 142622991, 'strand': 1}}]


In [18]:
lista_unicos = []
for symbol in lista_symbols:
    if symbol not in lista_unicos:
        lista_unicos.append(symbol)

In [22]:
resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
48 input query terms found no hit:	['SLC2A15', 'NONE', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 


In [24]:
# print(resultados_coords)

In [34]:
coords_genes = []
no_encontrados = []

for resultado in resultados_coords:
    
    gen = resultado.get("query")

    if resultado.get("notfound"):
        no_encontrados.append(gen)
        continue

    posicion = resultado.get("genomic_pos_hg19")
    
    if isinstance(posicion, list):
        posicion = posicion[0]

    if posicion:
        coords_genes.append({"gene_symbol": gen, "chr": str(posicion.get("chr")), "inicio": posicion.get("start"), "fin": posicion.get("end"), "cadena": posicion.get("strand")})



In [35]:
no_encontrados

['SLC2A15',
 'NONE',
 'LOC440311',
 '43160',
 'LOC100133091',
 'LOC101928978',
 'MIR7641-2',
 'LOC201175',
 'LOC101929163',
 'LOC339862',
 'LOC100287944',
 'BORCS6-ASMT',
 'LOC100505817',
 'LOC101927620',
 'FLJ45872',
 'FLJ25758',
 'LOC100129900',
 'LOC100131940',
 'LOC100288892',
 'FLJ23172',
 'KIAA1026',
 'LOC100129831',
 'LOC729305',
 'LOC391636',
 'LOC646114',
 'LOC100049717',
 'LOC646609',
 'LOC100288911',
 'LOC100287632',
 'LOC390800',
 'LOC729856',
 'LOC728276',
 'LOC100129138',
 'LOC101929066',
 'LOC100128880',
 'FLJ43860',
 'LOC645177',
 'LOC100507657',
 'LOC100130911',
 'LOC729160',
 'FLJ35379',
 'LOC100132423',
 'LOC391040',
 'DKFZp761B107',
 'LOC284930',
 'LOC100129620',
 'LOC646218',
 'LOC108783654']